In [1]:
# Ансамбль CatBoost, LightGBM и GRU

In [2]:
import numpy as np
import pandas as pd
from sklearn.metrics import root_mean_squared_log_error


catboost_log = pd.read_csv(
    "validation_predictions/catboost_prediction_log.csv"
).iloc[:, 0].to_numpy()

lgbm_log = pd.read_csv(
    "validation_predictions/lgbm_prediction_log.csv").iloc[:, 0].to_numpy()
gru_data = pd.read_csv(
    "validation_predictions/gru_validation_2025-12-15.csv"
)

gru_log = gru_data[
    "gru_prediction_log"
].to_numpy()

target = gru_data["target"].to_numpy()


assert len(catboost_log) == len(gru_log)


results = []

for alpha in np.arange(0, 1.01, 0.001):
    prediction_log = (
        alpha * catboost_log
        + (1 - alpha) * gru_log
    )

    prediction = np.expm1(prediction_log)

    score = root_mean_squared_log_error(
        target,
        prediction,
    )

    results.append({
        "alpha": alpha,
        "RMSLE": score,
    })


results = pd.DataFrame(results)
results = results.sort_values("RMSLE")

results.head(10)


,alpha,RMSLE
849,0.849,1.740573
848,0.848,1.740573
850,0.850,1.740573
847,0.847,1.740573
851,0.851,1.740573
846,0.846,1.740573
852,0.852,1.740574
845,0.845,1.740574
853,0.853,1.740574
844,0.844,1.740574


In [3]:
results_2=[]
for alpha in np.arange(0, 1.01, 0.01):
    prediction_log = (
        alpha * catboost_log
        + (1 - alpha) * lgbm_log
    )

    prediction = np.expm1(prediction_log)

    score = root_mean_squared_log_error(
        target,
        prediction,
    )

    results_2.append({
        "alpha": alpha,
        "RMSLE": score,
    })

In [4]:
results_2 = pd.DataFrame(results_2)
results_2 = results_2.sort_values("RMSLE")

results_2.head(10)

,alpha,RMSLE
67,0.67,1.740789
66,0.66,1.740790
68,0.68,1.740790
65,0.65,1.740790
69,0.69,1.740790
64,0.64,1.740792
70,0.70,1.740792
63,0.63,1.740794
71,0.71,1.740794
62,0.62,1.740796


In [5]:
subm_lg = pd.read_csv('submissions/submission_lgbm.csv')
subm_cat = pd.read_csv('submissions/submission_catboost.csv')
subm_gru = pd.read_csv('competition_predictions/simple_gru_competition_predictions.csv')


In [6]:
submission_ans_cat_lg = pd.DataFrame({
    "user_id": subm_lg["user_id"],
    "predict": subm_cat["predict"]*0.67+subm_lg["predict"]*(1-0.67),
})

submission_ans_cat_lg.to_csv("submissions/submission_catboost_lgbm.csv", index=False)


In [7]:

submission_ans_cat_gru = pd.DataFrame({
    "user_id": subm_lg["user_id"],
    "predict": subm_gru["gru_prediction"]*(1-0.849)+subm_cat["predict"]*0.849,
})

submission_ans_cat_gru.to_csv("submissions/submission_catboost_gru.csv", index=False)

In [8]:
from scipy.optimize import minimize


def find_best_weights(target, catboost_log, lgbm_log, gru_log):
    target_log = np.log1p(target)

    predictions = np.column_stack([
        catboost_log,
        lgbm_log,
        gru_log,
    ])

    def ensemble_rmsle(weights):
        prediction_log = predictions @ weights
        return np.sqrt(np.mean((target_log - prediction_log) ** 2))

    result = minimize(
        ensemble_rmsle,
        x0=[0.7, 0.2, 0.1],
        bounds=[(0, 1), (0, 1), (0, 1)],
        constraints={
            'type': 'eq',
            'fun': lambda weights: weights.sum() - 1,
        },
        method='SLSQP',
    )

    return result.x, result.fun


In [9]:
best_weights, best_rmsle = find_best_weights(
    target,
    catboost_log,
    lgbm_log,
    gru_log,
)

catboost_weight = best_weights[0]
lgbm_weight = best_weights[1]
gru_weight = best_weights[2]

print('CatBoost weight:', catboost_weight)
print('LightGBM weight:', lgbm_weight)
print('GRU weight:', gru_weight)
print('Validation RMSLE:', best_rmsle)


CatBoost weight: 0.67496853430468
LightGBM weight: 0.18943503286893917
GRU weight: 0.13559643282638076
Validation RMSLE: 1.7404279365421538


In [10]:
assert subm_cat['user_id'].equals(subm_lg['user_id'])
assert subm_cat['user_id'].equals(subm_gru['user_id'])

catboost_submit_log = np.log1p(subm_cat['predict'].to_numpy())
lgbm_submit_log = np.log1p(subm_lg['predict'].to_numpy())
gru_submit_log = subm_gru['gru_prediction_log'].to_numpy()

final_prediction_log = (
    catboost_weight * catboost_submit_log
    + lgbm_weight * lgbm_submit_log
    + gru_weight * gru_submit_log
)

final_submission = pd.DataFrame({
    'user_id': subm_cat['user_id'],
    'predict': np.expm1(final_prediction_log),
})

final_submission.to_csv(
    'submissions/submission_catboost_lgbm_gru.csv',
    index=False,
)

final_submission.head()


,user_id,predict
0,2,2.156077
1,7,87.202674
2,15,8.375667
3,18,135.254392
4,23,0.469241
